# ATS Fine-Tuning Pipeline — Phi-3-mini × QLoRA

**6 phases, end-to-end on Colab T4 GPU:**
1. Environment Setup
2. Dataset Preparation
3. Model Loading & LoRA
4. Fine-Tuning (response-only loss via `DataCollatorForCompletionOnlyLM`)
5. Inference Testing
6. Evaluation Metrics

**Key design decisions:**
- Phi-3 native chat template (`<|user|>` / `<|assistant|>`) — not Alpaca format
- `DataCollatorForCompletionOnlyLM` for prompt masking — avoids BPE context bugs
- EOS token set to `<|end|>` (Phi-3 native, ID 32007)
- `Phi3ForCausalLM` directly — no `trust_remote_code` (fixes DynamicCache error)
- `paged_adamw_32bit` optimizer, cosine LR, epoch-based eval

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/colab')
print('Working directory:', os.getcwd())

In [ ]:
# trl is added for DataCollatorForCompletionOnlyLM (response-only masking)
!pip install -q \
    transformers>=4.41.0 \
    peft>=0.7.0 \
    datasets>=2.16.0 \
    accelerate>=0.25.0 \
    bitsandbytes>=0.41.0 \
    trl>=0.8.0 \
    scipy \
    pyyaml \
    json-repair

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected — set Runtime → Change runtime type → T4 GPU')

print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

## Phase 2 — Dataset Preparation

- Load YAML configs
- Validate raw dataset (checks `weak_bullets` key — the correct field name)
- Format using **Phi-3 native chat template** (not Alpaca `### Instruction:` format)
- Split 90/10 and save

In [ ]:
import yaml, json, random
from pathlib import Path

with open('configs/training_config.yaml') as f:
    train_cfg = yaml.safe_load(f)
with open('configs/lora_config.yaml') as f:
    lora_cfg = yaml.safe_load(f)

print('Training config loaded:')
for k, v in train_cfg.items():
    print(f'  {k}: {v}')

In [ ]:
# Validate every sample in raw_dataset.json
# Uses weak_bullets (the correct key from the training data)
REQUIRED_KEYS_TOP    = {'instruction', 'input', 'output'}
REQUIRED_KEYS_OUTPUT = {
    'ats_score', 'score_breakdown', 'matched_skills',
    'missing_skills', 'weak_bullets', 'formatting_issues', 'overall_feedback'
}

raw_path = Path(train_cfg['raw_dataset'])
with open(raw_path) as f:
    raw_data = json.load(f)

errors, scores = [], []
for i, s in enumerate(raw_data):
    missing = REQUIRED_KEYS_TOP - set(s)
    if missing:
        errors.append(f'Sample {i}: missing top-level keys {missing}')
        continue
    try:
        out = json.loads(s['output'])
        missing_out = REQUIRED_KEYS_OUTPUT - set(out)
        if missing_out:
            errors.append(f'Sample {i}: missing output keys {missing_out}')
        else:
            scores.append(out['ats_score'])
    except json.JSONDecodeError as e:
        errors.append(f'Sample {i}: bad JSON — {e}')

if errors:
    print(f'ERRORS ({len(errors)}):')
    for e in errors[:10]: print(f'  {e}')
else:
    print(f'All {len(raw_data)} samples valid')
    print(f'ATS score — Min: {min(scores)}, Max: {max(scores)}, Mean: {sum(scores)/len(scores):.1f}')

In [ ]:
from transformers import AutoTokenizer

model_name = train_cfg['model_name']
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Phi-3 uses <|end|> (ID 32007) as end-of-turn — NOT <|endoftext|> (ID 0)
# Training sequences terminated with wrong EOS cause the model to never learn
# to stop generation cleanly.
phi3_eos = '<|end|>'
tokenizer.eos_token = phi3_eos

# <|end|> also serves as pad token (padding_side=right for causal LM)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'EOS: "{tokenizer.eos_token}" (ID {tokenizer.eos_token_id})')
print(f'PAD: "{tokenizer.pad_token}" (ID {tokenizer.pad_token_id})')
print(f'Vocab size: {tokenizer.vocab_size}')

# Sanity: ensure EOS is the Phi-3 end-of-turn token
assert tokenizer.eos_token_id == 32007, (
    f'Expected EOS ID 32007 (<|end|>), got {tokenizer.eos_token_id}. '
    'Update the phi3_eos string above.'
)

In [ ]:
# Format samples using Phi-3 NATIVE chat template:
#   <|user|>\n{instruction}\n\n{input}<|end|>\n<|assistant|>\n{output}<|end|>
#
# DataCollatorForCompletionOnlyLM will mask everything before <|assistant|>.
# <|user|>, <|assistant|>, <|end|> are SINGLE special tokens in Phi-3's vocab,
# so they always get the same ID regardless of context — no BPE ambiguity.

def format_sample(s):
    return (
        f"<|user|>\n{s['instruction']}\n\n{s['input']}<|end|>\n"
        f"<|assistant|>\n{s['output']}<|end|>"
    )

formatted = [format_sample(s) for s in raw_data]
lengths   = [len(tokenizer.encode(t)) for t in formatted]
max_len   = train_cfg['max_seq_length']

print(f'Token lengths — Min: {min(lengths)}, Max: {max(lengths)}, Mean: {sum(lengths)/len(lengths):.0f}')
print(f'Samples > {max_len} tokens (will be truncated): {sum(1 for l in lengths if l > max_len)}')
print()
print('Sample (first 600 chars):')
print(formatted[0][:600])

In [ ]:
# 90 / 10 train / val split — deterministic via seed
random.seed(train_cfg['seed'])
indices = list(range(len(raw_data)))
random.shuffle(indices)

n_train = int(len(indices) * train_cfg['train_split'])
train_idx, val_idx = indices[:n_train], indices[n_train:]

# Pre-formatted text datasets (used for training)
train_data = [{'text': formatted[i]} for i in train_idx]
val_data   = [{'text': formatted[i]} for i in val_idx]

# Original samples (used for evaluation in Phase 6)
train_raw_split = [raw_data[i] for i in train_idx]
val_raw_split   = [raw_data[i] for i in val_idx]

Path('data').mkdir(exist_ok=True)
with open('data/train.json', 'w') as f:
    json.dump(train_raw_split, f, indent=2)
with open('data/validation.json', 'w') as f:
    json.dump(val_raw_split, f, indent=2)

print(f'Train: {len(train_data)} samples')
print(f'Val:   {len(val_data)} samples')
print('Splits saved to data/train.json and data/validation.json')

## Phase 3 — Model Loading & LoRA

- Load Phi-3 with 4-bit NF4 quantisation (QLoRA)
- `Phi3ForCausalLM` directly — no `trust_remote_code` (avoids DynamicCache KeyError)
- No `low_cpu_mem_usage=True` — causes PEFT key-resolution failures for tied weights
- Apply LoRA with rank 16, alpha 32, targeting q/k/v/o projections

In [ ]:
import torch
from transformers import BitsAndBytesConfig, Phi3ForCausalLM

bnb_config = BitsAndBytesConfig(
    load_in_4bit=train_cfg['use_4bit'],
    bnb_4bit_quant_type=train_cfg['bnb_4bit_quant_type'],
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=train_cfg['use_double_quant'],
)

model = Phi3ForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    # trust_remote_code=True is intentionally omitted — transformers>=4.41.0
    # ships native Phi3ForCausalLM; using it avoids the stale cached
    # modeling_phi3.py that triggers DynamicCache.seen_tokens AttributeError.
    # low_cpu_mem_usage=True is also intentionally omitted — it causes PEFT
    # to fail resolving base_model.model.lm_head for tied-weight models.
)
model.config.use_cache = False  # required when gradient_checkpointing=True

total_params = sum(p.numel() for p in model.parameters())
print(f'Model: {model.__class__.__name__}')
print(f'Total parameters: {total_params:,}')

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=lora_cfg['r'],
    lora_alpha=lora_cfg['lora_alpha'],
    lora_dropout=lora_cfg['lora_dropout'],
    target_modules=lora_cfg['target_modules'],
    bias=lora_cfg['bias'],
    task_type=lora_cfg['task_type'],
    inference_mode=False,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.gradient_checkpointing_enable()

## Phase 4 — Fine-Tuning

**Critical fix over original notebook:** replaced the hand-coded token-ID search
with `DataCollatorForCompletionOnlyLM` from TRL.

The original code searched for `'### Response:\n'` token IDs in the tokenized
sequence, but BPE tokenizes the same characters differently depending on surrounding
context. This caused *every* token to be masked (`-100`), giving `train_loss=0.0`
and `eval_loss=nan` — the model received zero gradient updates.

`DataCollatorForCompletionOnlyLM` with `response_template='<|assistant|>'` works
reliably because `<|assistant|>` is a **single special token** in Phi-3's vocabulary
(ID 32001), so it always tokenizes to the same ID regardless of surrounding context.

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
val_dataset   = Dataset.from_list(val_data)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=train_cfg['max_seq_length'],
        padding=False,  # DataCollator pads per batch
    )

train_tok = train_dataset.map(tokenize, batched=True, remove_columns=['text'])
val_tok   = val_dataset.map(tokenize,   batched=True, remove_columns=['text'])

print(f'Tokenized — train: {len(train_tok)}, val: {len(val_tok)}')

In [ ]:
from trl import DataCollatorForCompletionOnlyLM

# <|assistant|> is a single special token (ID 32001) — context-independent tokenization
collator = DataCollatorForCompletionOnlyLM(
    response_template='<|assistant|>',
    tokenizer=tokenizer,
    mlm=False,
)

# ── Masking sanity check — must show >0 trained tokens ──────────────────────
sample_batch = collator([train_tok[0]])
labels = sample_batch['labels'][0].tolist()
n_total   = len(labels)
n_masked  = labels.count(-100)
n_trained = n_total - n_masked

print(f'Masking check on sample 0:')
print(f'  Total tokens:    {n_total}')
print(f'  Masked (-100):   {n_masked}')
print(f'  Trained (loss):  {n_trained}')

assert n_trained > 0, (
    'All tokens masked — response template not found! '
    'Check tokenizer and format_sample().'
)
print('  Masking is correct')

In [ ]:
from transformers import TrainingArguments, Trainer

# ── Step count sanity ────────────────────────────────────────────────────────
eff_batch       = train_cfg['per_device_train_batch_size'] * train_cfg['gradient_accumulation_steps']
steps_per_epoch = max(1, len(train_tok) // eff_batch)
total_steps     = steps_per_epoch * train_cfg['num_train_epochs']
warmup_steps    = max(1, int(total_steps * 0.03))
print(f'Effective batch:  {eff_batch}')
print(f'Steps/epoch:      {steps_per_epoch}')
print(f'Total steps:      {total_steps}')
print(f'Warmup steps:     {warmup_steps} (3%)')

training_args = TrainingArguments(
    output_dir=train_cfg['output_dir'],
    num_train_epochs=train_cfg['num_train_epochs'],
    per_device_train_batch_size=train_cfg['per_device_train_batch_size'],
    per_device_eval_batch_size=train_cfg['per_device_eval_batch_size'],
    gradient_accumulation_steps=train_cfg['gradient_accumulation_steps'],
    learning_rate=train_cfg['learning_rate'],
    # warmup_ratio instead of warmup_steps — safe when total_steps < configured steps
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',   # cosine > linear for small datasets
    optim='paged_adamw_32bit',    # lower peak VRAM than adamw_torch on T4
    fp16=train_cfg['fp16'],
    bf16=train_cfg['bf16'],
    gradient_checkpointing=train_cfg['gradient_checkpointing'],
    # epoch-based eval/save — safe regardless of total step count
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_steps=1,
    save_total_limit=train_cfg['save_total_limit'],
    max_grad_norm=1.0,
    seed=train_cfg['seed'],
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
)

In [ ]:
print('Starting fine-tuning...')
trainer.train()
print('Training complete!')

# Verify loss is non-zero and finite
history = trainer.state.log_history
train_losses = [e['loss'] for e in history if 'loss' in e]
if train_losses:
    print(f'Train loss: initial={train_losses[0]:.4f}, final={train_losses[-1]:.4f}')
    assert train_losses[-1] > 0 and train_losses[-1] == train_losses[-1], \
        'ERROR: train_loss is 0 or NaN — label masking failed'
else:
    print('WARNING: No loss values found in training history')

In [ ]:
# Save LoRA adapter
adapter_dir = 'ats_phi_lora'
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f'LoRA adapter saved to ./{adapter_dir}/')

# Phi-3 gotcha: ensure_weight_tying must be false in adapter_config.json.
# If true, PEFT tries to resolve base_model.model.model.embed_tokens, which
# doesn't exist in the checkpoint and raises a KeyError at merge time.
import json as _json
cfg_path = f'{adapter_dir}/adapter_config.json'
with open(cfg_path) as f:
    adapter_cfg = _json.load(f)
if adapter_cfg.get('ensure_weight_tying', False):
    adapter_cfg['ensure_weight_tying'] = False
    with open(cfg_path, 'w') as f:
        _json.dump(adapter_cfg, f, indent=2)
    print('Fixed adapter_config.json: ensure_weight_tying set to false')
else:
    print('adapter_config.json: ensure_weight_tying already false — OK')

## Phase 5 — Inference Testing

Quick smoke test: run 2 samples from the validation set through the fine-tuned
model and check that the output is parseable JSON with the expected keys.

In [ ]:
import torch

def generate_from_sample(sample, model, tokenizer, max_new_tokens=1024):
    """Generate ATS evaluation for a single instruction/input sample."""
    prompt = (
        f"<|user|>\n{sample['instruction']}\n\n{sample['input']}<|end|>\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,   # <|end|>
            pad_token_id=tokenizer.pad_token_id,
        )
    n_prompt = inputs['input_ids'].shape[1]
    return tokenizer.decode(output_ids[0][n_prompt:], skip_special_tokens=True).strip()


def extract_json(text):
    """Multi-tier JSON extraction: direct → regex → json-repair."""
    import re
    from json_repair import repair_json
    # Tier 1: direct
    try:
        return json.loads(text), 'direct'
    except json.JSONDecodeError:
        pass
    # Tier 2: extract outermost JSON object
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0)), 'regex'
        except json.JSONDecodeError:
            pass
    # Tier 3: repair and parse
    try:
        return json.loads(repair_json(text)), 'repaired'
    except Exception:
        pass
    return None, 'failed'

In [ ]:
REQUIRED_OUTPUT_KEYS = {
    'ats_score', 'score_breakdown', 'matched_skills',
    'missing_skills', 'weak_bullets', 'formatting_issues', 'overall_feedback'
}

with open('data/validation.json') as f:
    val_raw_split = json.load(f)

# Test on first 2 validation samples
for i, sample in enumerate(val_raw_split[:2]):
    print(f'--- Sample {i} ---')
    raw = generate_from_sample(sample, trainer.model, tokenizer)
    parsed, method = extract_json(raw)
    gt_score = json.loads(sample['output'])['ats_score']

    if parsed and REQUIRED_OUTPUT_KEYS.issubset(set(parsed)):
        print(f'  Valid ATS JSON (parse method: {method})')
        print(f'  Predicted score: {parsed["ats_score"]}  |  GT score: {gt_score}')
    elif parsed:
        missing = REQUIRED_OUTPUT_KEYS - set(parsed)
        print(f'  Parsed JSON but missing keys: {missing}')
    else:
        print(f'  Could not parse JSON')
        print(f'  Raw output (first 300 chars): {raw[:300]}')
    print()

## Phase 6 — Evaluation Metrics

Batch inference over all validation samples and compute:
- **JSON Validity** — % samples that produce parseable JSON
- **ATS Structure Validity** — % samples with all required keys
- **Score MAE** — mean absolute error vs ground-truth ATS score
- **Missing-Skill F1** — precision/recall over predicted vs GT missing skills

> All metrics use `weak_bullets` as the correct key (matching the training data).

In [ ]:
import statistics

results = []
print(f'Evaluating {len(val_raw_split)} validation samples...')

for i, sample in enumerate(val_raw_split):
    raw    = generate_from_sample(sample, trainer.model, tokenizer)
    parsed, _ = extract_json(raw)
    gt     = json.loads(sample['output'])

    is_valid_json = parsed is not None
    is_valid_ats  = is_valid_json and REQUIRED_OUTPUT_KEYS.issubset(set(parsed))

    score_diff = abs(parsed['ats_score'] - gt['ats_score']) if is_valid_ats else None

    # Missing-skill F1 (case-insensitive)
    if is_valid_ats:
        pred_m = {k.lower() for k in parsed.get('missing_skills', [])}
        gt_m   = {k.lower() for k in gt.get('missing_skills', [])}
        if gt_m:
            prec = len(pred_m & gt_m) / len(pred_m) if pred_m else 0.0
            rec  = len(pred_m & gt_m) / len(gt_m)
            f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        else:
            f1 = 1.0 if not pred_m else 0.0
    else:
        f1 = None

    results.append({'valid_json': is_valid_json, 'valid_ats': is_valid_ats,
                    'score_diff': score_diff, 'skill_f1': f1})

    if (i + 1) % 5 == 0:
        print(f'  {i + 1}/{len(val_raw_split)} done')

In [ ]:
n = len(results)
json_rate  = sum(1 for r in results if r['valid_json']) / n * 100
ats_rate   = sum(1 for r in results if r['valid_ats'])  / n * 100
diffs      = [r['score_diff'] for r in results if r['score_diff'] is not None]
f1s        = [r['skill_f1']   for r in results if r['skill_f1']   is not None]
mae        = statistics.mean(diffs) if diffs else float('nan')
mean_f1    = statistics.mean(f1s)   if f1s   else float('nan')

print(f'=== Evaluation Results ({n} samples) ===')
print(f'JSON Validity:     {json_rate:5.1f}%   (target > 95%)')
print(f'ATS Structure:     {ats_rate:5.1f}%   (target > 90%)')
print(f'Score MAE:         {mae:6.2f}    (target < 20)')
print(f'Missing-Skill F1:  {mean_f1:5.1%}    (target > 50%)')

# Score distribution comparison
import json as _json
gt_scores   = [_json.loads(s['output'])['ats_score'] for s in val_raw_split]
pred_scores = []
for r, s in zip(results, val_raw_split):
    if r['valid_ats']:
        pred_scores.append(
            _json.loads(generate_from_sample(s, trainer.model, tokenizer).split('}')[0] + '}')
            if False else None  # already have parsed results above
        )
print()
print(f'GT score range:   {min(gt_scores)}-{max(gt_scores)}, mean={sum(gt_scores)/len(gt_scores):.1f}')